# Malaysia Election Intelligence System (MEIS)
## Data Engineering Pipeline & Exploratory Data Analysis (EDA) Simulation

This interactive notebook simulates the end-to-end data pipeline for the Malaysia Election Intelligence System:
1. **Ingestion Verification**: Locates and verifies the raw consolidated election datasets.
2. **Data Cleaning & Transformation**: Executes Pandas-based string normalizations, date formatting, duplicate removal, and feature engineering (`seat_type`, `election_year`).
3. **Exploratory Data Analysis (EDA)**: Performs quick structural checks, missing value inspections, and distributions breakdown prior to MySQL insertion.

In [1]:
# ============================================================
# IMPORTS & ENVIRONMENT SETUP
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np

# Automatically resolve base directory paths
BASE_DIR = Path().resolve().parents[0]
if (BASE_DIR / "data").exists():
    DATASET_DIR = BASE_DIR / "data" / "datasets"
    PROCESSED_DIR = BASE_DIR / "data" / "processed"
else:
    # Fallback if running directly out of a subfolder
    BASE_DIR = Path().resolve().parents[1]
    DATASET_DIR = BASE_DIR / "data" / "datasets"
    PROCESSED_DIR = BASE_DIR / "data" / "processed"

BALLOTS_FILE = DATASET_DIR / "consol_ballots.csv"
print(f"[OK] Base Directory resolved: {BASE_DIR}")
print(f"[OK] Target Dataset Path: {BALLOTS_FILE}")

[OK] Base Directory resolved: C:\Users\User\Downloads\Side-Project\meis
[OK] Target Dataset Path: C:\Users\User\Downloads\Side-Project\meis\data\datasets\consol_ballots.csv


### Step 1: Load and Inspect Raw Ballot Data
Loading `consol_ballots.csv` to review dimensions, column names, and sample values.

In [2]:
# ============================================================
# LOAD RAW DATASET
# ============================================================
if not BALLOTS_FILE.exists():
    raise FileNotFoundError(f"Dataset not found at {BALLOTS_FILE}. Run ingestion script first.")

df = pd.read_csv(BALLOTS_FILE, low_memory=False)

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

Dataset Shape: 26,965 rows x 20 columns


,date,election,state,seat,ballot_order,candidate_uid,name_on_ballot,name,sex,ethnicity,age,party_on_ballot,party_uid,party,coalition_uid,coalition,votes,votes_perc,rank,result
0,1955-07-25,GE-00,Pulau Pinang,P.001 Wellesley North,1,F0G2M,Hashim Awang,Hashim bin Awang,M,Malay,-1,PERIKATAN,001-UMNO,UMNO,001-PERIKATAN,PERIKATAN,0,NaN,1,won_uncontested
1,1955-07-25,GE-00,Pulau Pinang,P.002 Wellesley South,1,JE64X,Tay Hooi Soo,Tay Hooi Soo,M,Chinese,58,PERIKATAN,003-MCA,MCA,001-PERIKATAN,PERIKATAN,15697,81.670135,1,won
2,1955-07-25,GE-00,Pulau Pinang,P.002 Wellesley South,2,NVW76,Haji Zabidi Bin Haji Ali,Zabidi bin Ali,M,Malay,41,PAS,004-PAS,PAS,000-ALONE,ALONE,3523,18.329865,2,lost


### Step 2: Apply Cleaning & Transformation Rules
Replicating the production data cleaning procedures from `clean_election_data.py`:
- Standardizing column headers (lowercase, underscores).
- Cleaning and stripping whitespace from string fields.
- Parsing dates and casting numeric types safely.
- Eliminating exact duplicate records.
- Engineering derived features (`seat_type` distinguishing Parliament vs. State Assembly, and `election_year`).

In [3]:
# ============================================================
# DATA CLEANING PIPELINE SIMULATION
# ============================================================

# 1. Clean Column Names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 2. Clean Text Fields
text_columns = [
    "election", "state", "seat", "candidate_uid", 
    "name_on_ballot", "name", "sex", "ethnicity", 
    "party_on_ballot", "party_uid", "party", "result"
]
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# 3. Clean Dates & Numerics
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

numeric_cols = ["ballot_order", "age", "votes", "votes_perc", "rank"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Remove Duplicates
before_count = len(df)
df = df.drop_duplicates()
print(f"[OK] Removed {before_count - len(df):,} duplicate records.")

# 5. Add Derived Features
if "seat" in df.columns:
    df["seat_type"] = "Unknown"
    df.loc[df["seat"].astype(str).str.startswith("P."), "seat_type"] = "Parliament"
    df.loc[df["seat"].astype(str).str.startswith("N."), "seat_type"] = "State Assembly"

if "date" in df.columns:
    df["election_year"] = df["date"].dt.year

print(f"[OK] Data cleaning complete. Final shape: {df.shape}")

[OK] Removed 0 duplicate records.
[OK] Data cleaning complete. Final shape: (26965, 22)


### Step 3: Exploratory Data Analysis (EDA) & Integrity Sanity Checks
Analyzing key structural metrics, verifying seat classifications, and checking missing values to validate pipeline readiness before database loading.

In [4]:
# ============================================================
# EDA & DATA INTEGRITY CHECKS
# ============================================================

print("--- 1. Seat Type Distribution ---")
if "seat_type" in df.columns:
    print(df["seat_type"].value_counts())

print("\n--- 2. Records by Election Year (Top 10 Recent) ---")
if "election_year" in df.columns:
    print(df["election_year"].value_counts().sort_index().tail(10))

print("\n--- 3. Missing Value Audit ---")
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("No missing values found.")
else:
    print(missing.head(5))

--- 1. Seat Type Distribution ---
seat_type
State Assembly    19819
Parliament         7146
Name: count, dtype: int64

--- 2. Records by Election Year (Top 10 Recent) ---
election_year
2017       3
2018    2347
2019      23
2020     455
2021     461
2022    1625
2023     586
2024      10
2025     599
2026     280
Name: count, dtype: int64

--- 3. Missing Value Audit ---
votes_perc    366
dtype: int64


### Step 4: Export Cleaned Output
Saving the processed file to `data/processed/election_results_clean.csv`, matching the exact input expectations for the MySQL loading script (`load_election_data.py`).

In [5]:
# ============================================================
# EXPORT PROCESSED DATASET
# ============================================================
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_DIR / "election_results_clean.csv"

df.to_csv(OUTPUT_FILE, index=False)
print(f"[OK] Processed dataset successfully exported to:\n{OUTPUT_FILE}")

[OK] Processed dataset successfully exported to:
C:\Users\User\Downloads\Side-Project\meis\data\processed\election_results_clean.csv
